*0.4 Deep learning basics*

# Training loop from scratch

**The situation.** You need a sentiment model for product reviews and the team wants to see the whole training process, not a `.fit()` that hides it — because next month they will fine-tune an LLM, and that is the same loop with a bigger model.

**The loop.** For each epoch: shuffle the data; for each batch: forward → loss → `zero_grad` → `backward` → `optimizer.step()`. After each epoch: evaluate on validation data in `eval` mode without gradients. That is all of it. Everything else — schedulers, mixed precision, checkpoints — is added around this skeleton.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Real data.** SST-2 movie-review sentences, tokenized with BERT's tokenizer (a production tokenizer; the model is our own). 8,000 training sentences keep the notebook quick.

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# SST-2: 67k movie-review sentences labelled positive/negative — the standard small sentiment set.
sst2 = load_dataset("stanfordnlp/sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def encode(rows, max_length=32):
    encoded = tokenizer(
        rows["sentence"], truncation=True, max_length=max_length, padding="max_length"
    )
    return {"ids": encoded["input_ids"], "label": rows["label"]}


train_rows = sst2["train"].shuffle(seed=0).select(range(8000)).map(encode, batched=True)
val_rows = sst2["validation"].map(encode, batched=True)
train_ids = torch.tensor(train_rows["ids"])
train_labels = torch.tensor(train_rows["label"])
val_ids = torch.tensor(val_rows["ids"])
val_labels = torch.tensor(val_rows["label"])
print(
    "train:",
    tuple(train_ids.shape),
    "| validation:",
    tuple(val_ids.shape),
    "| vocabulary:",
    tokenizer.vocab_size,
)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

train: (8000, 32) | validation: (872, 32) | vocabulary: 30522


**The model and the loop.** The `SentimentClassifier` from the previous item; the loop written out in full.

In [3]:
import time

import torch.nn.functional as F
from torch import nn


class SentimentClassifier(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_size: int = 64, classes: int = 2):
        super().__init__()
        self.embedding = nn.EmbeddingBag(
            vocabulary_size, embedding_size, mode="mean", padding_idx=0
        )
        self.hidden = nn.Linear(embedding_size, 64)
        self.output = nn.Linear(64, classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, token_ids):
        return self.output(self.dropout(F.relu(self.hidden(self.embedding(token_ids)))))


torch.manual_seed(0)
model = SentimentClassifier(tokenizer.vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
batch_size = 64

started = time.perf_counter()
for epoch in range(1, 6):
    model.train()  # 1. train mode: dropout on
    order = torch.randperm(len(train_ids))  # 2. shuffle
    total_loss = 0.0
    for start in range(0, len(order), batch_size):
        batch = order[start : start + batch_size]
        logits = model(train_ids[batch])  # 3. forward
        loss = F.cross_entropy(logits, train_labels[batch])  # 4. loss
        optimizer.zero_grad()  # 5. clear old gradients
        loss.backward()  # 6. gradients
        optimizer.step()  # 7. update weights
        total_loss += loss.item() * len(batch)

    model.eval()  # 8. evaluate: dropout off, no gradients
    with torch.no_grad():
        val_accuracy = (model(val_ids).argmax(dim=1) == val_labels).float().mean().item()
    print(
        
            f"epoch {epoch}  train loss {total_loss / len(order):.3f}  val accuracy "
            f"{val_accuracy:.1%}"
        
    )
print(f"trained in {time.perf_counter() - started:.0f} s on CPU")
assert val_accuracy > 0.7

epoch 1  train loss 0.672  val accuracy 64.3%


epoch 2  train loss 0.571  val accuracy 71.8%


epoch 3  train loss 0.419  val accuracy 75.8%


epoch 4  train loss 0.298  val accuracy 75.3%


epoch 5  train loss 0.219  val accuracy 76.4%
trained in 2 s on CPU


**Reading the output.** Training loss falls every epoch; validation accuracy climbs to around 75–80% from a model that averages word vectors — in well under a minute on a laptop CPU. When training loss keeps falling but validation stops improving, that is the overfitting from 0.2, and the epoch to stop at.

```
epoch ─▶ shuffle ─▶ batch ─▶ forward ─▶ loss ─▶ zero_grad ─▶ backward ─▶ step ─▶ next batch
                                                                          │
                                                        end of epoch ─▶ eval(): validation accuracy
```

**The rule to remember.** Forward, loss, zero, backward, step. Evaluate in `eval` mode under `no_grad`. Every training script — including LLM fine-tuning — is this loop with more around it.

| Use it when | Don't when | Instead use |
|---|---|---|
| you need control or understanding: custom losses, research, learning | standard fine-tuning of a Hugging Face model | `transformers.Trainer` / `trl` (the same loop, with the extras built in) |

**Watch out**
- `loss.item()` every step syncs the GPU; on a GPU, accumulate the tensor and read it once per epoch.
- Evaluate on the validation set every epoch and keep the best checkpoint; the last epoch is rarely the best.
- Learning rate is the knob: if the loss jumps or goes to `nan`, lower it first.